In [1]:
import torch  # PyTorch library (used for building and running the model)

# Check if a GPU is available (important for faster training)
print("CUDA available:", torch.cuda.is_available())

# If GPU is available, print the name of the GPU being used
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

from google.colab import drive  # Allows access to Google Drive files

# Mount Google Drive so we can load data and save models
drive.mount("/content/drive")

CUDA available: True
GPU: Tesla T4
Mounted at /content/drive


In [ ]:
# == Copying data to local colab storage

import os

# Base folder in Google Drive where your project files are stored
BASE_PATH = "/content/drive/MyDrive/baseball"

# Paths to data stored in Google Drive
DRIVE_FRAMES_DIR = f"{BASE_PATH}/extracted_frames" # extracted image frames
DRIVE_XML_DIR = f"{BASE_PATH}/annotations" # XML annotation files

# Local (temporary) paths inside Colab runtime
# Using local paths makes training faster than reading directly from Drive
LOCAL_FRAMES_DIR = "/content/extracted_frames"
LOCAL_XML_DIR = "/content/annotations"

# Remove any existing local copies (to avoid duplicates or stale files)
!rm -rf /content/extracted_frames /content/annotations

# Copy data from Google Drive to local Colab storage (makes training faster)
!cp -r "$DRIVE_FRAMES_DIR" "$LOCAL_FRAMES_DIR"
!cp -r "$DRIVE_XML_DIR" "$LOCAL_XML_DIR"

# Print number of files copied to confirm everything worked
print("Local frames:", len(os.listdir(LOCAL_FRAMES_DIR)))
print("Local annotations:", len(os.listdir(LOCAL_XML_DIR)))

Local frames: 577
Local annotations: 44


In [ ]:
%%writefile faster_rcnn_baseball.py
# ===== CELL 3: FASTER R-CNN SCRIPT =====

# Libraries for file handling, math, images, and deep learning
import os
import time
import random
import numpy as np
import cv2
import torch
import torchvision
import xml.etree.ElementTree as ET
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor


# data augmentation
# Adds variation to training images (helps generalization)
def augment_image_and_box(image, box):

    h, w = image.shape[:2]  # image height/width

    # Random brightness / contrast adjustment
    if random.random() < 0.50:
        alpha = random.uniform(0.80, 1.20) # contrast
        beta = random.uniform(-20, 20) # brightness
        image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)

    # Random slight blur
    if random.random() < 0.25:
        image = cv2.GaussianBlur(image, (3, 3), 0)

    # Random small rotation
    if random.random() < 0.30:
        angle = random.uniform(-5, 5)

        # Rotation matrix centered on image
        cx_img, cy_img = w / 2, h / 2
        matrix = cv2.getRotationMatrix2D((cx_img, cy_img), angle, 1.0)

        # Rotate image
        rotated = cv2.warpAffine(
            image,
            matrix,
            (w, h),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(0, 0, 0)
        )

        # Adjust bounding box after rotation
        xtl, ytl, xbr, ybr = box

        corners = np.array([
            [xtl, ytl],
            [xbr, ytl],
            [xbr, ybr],
            [xtl, ybr]
        ], dtype=np.float32)

        ones = np.ones((corners.shape[0], 1), dtype=np.float32)
        corners_h = np.hstack([corners, ones])

        new_corners = corners_h @ matrix.T

        # Get new bounding box coordinates
        new_xtl = np.clip(new_corners[:, 0].min(), 0, w - 1)
        new_ytl = np.clip(new_corners[:, 1].min(), 0, h - 1)
        new_xbr = np.clip(new_corners[:, 0].max(), 0, w - 1)
        new_ybr = np.clip(new_corners[:, 1].max(), 0, h - 1)

        image = rotated
        box = [new_xtl, new_ytl, new_xbr, new_ybr]

    return image, box


# Loads images and bounding boxes from XML files
class BaseballDetectionDataset(Dataset):

    def __init__(self, frames_dir, xml_dir, augment=False):
        self.frames_dir = frames_dir # folder with images
        self.xml_dir = xml_dir # folder with annotations
        self.augment = augment # whether to apply augmentation
        self.samples = [] # list of (image, box)

        self._build_index()

    def _build_index(self):
        # Read all XML annotation files
        xml_files = [f for f in os.listdir(self.xml_dir) if f.endswith(".xml")]

        for xml_file in xml_files:
            xml_path = os.path.join(self.xml_dir, xml_file)
            video_stem = os.path.splitext(xml_file)[0]

            tree = ET.parse(xml_path)
            root = tree.getroot()

            # Loop through each tracked object
            for track in root.findall("track"):
                if track.attrib.get("label") != "baseball":
                    continue

                for box in track.findall("box"):

                    # Skip frames where ball is not visible
                    if int(box.attrib.get("outside", "0")) == 1:
                        continue

                    # Only keep frames where ball is moving
                    moving_attr = box.find("attribute[@name='moving']")
                    if moving_attr is None or moving_attr.text.strip().lower() != "true":
                        continue

                    frame_num = int(box.attrib["frame"])

                    # Build image filename
                    image_name = f"{video_stem}_frame_{frame_num}.jpg"
                    image_path = os.path.join(self.frames_dir, image_name)

                    # Skip if image file doesn't exist
                    if not os.path.exists(image_path):
                        continue

                    # Get bounding box coordinates
                    xtl = float(box.attrib["xtl"])
                    ytl = float(box.attrib["ytl"])
                    xbr = float(box.attrib["xbr"])
                    ybr = float(box.attrib["ybr"])

                    # Skip invalid boxes
                    if xbr <= xtl or ybr <= ytl:
                        continue

                    # Save sample
                    self.samples.append({
                        "image_path": image_path,
                        "box": [xtl, ytl, xbr, ybr]
                    })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]

        # Load image
        image = cv2.imread(sample["image_path"])
        if image is None:
            raise RuntimeError(f"Could not read image: {sample['image_path']}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        box = sample["box"]

        # Apply augmentation only for training data
        if self.augment:
            image, box = augment_image_and_box(image, box)

        # Convert image to tensor
        image = F.to_tensor(image)

        # Convert box to tensor format
        box = torch.tensor([box], dtype=torch.float32)

        # Build dictionary
        target = {
            "boxes": box,
            "labels": torch.tensor([1], dtype=torch.int64),  # 1 = baseball
            "image_id": torch.tensor([idx]),
            "area": torch.tensor([(box[0, 2] - box[0, 0]) * (box[0, 3] - box[0, 1])]),
            "iscrowd": torch.tensor([0], dtype=torch.int64)
        }

        return image, target

# Required because each sample can have different structure
def collate_fn(batch):
    return tuple(zip(*batch))

# Load pretrained Faster R-CNN and adjust for baseball detection
def get_model(num_classes = 2):
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights = "DEFAULT")

    # Replace final classification layer
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model


# IoU CALCULATION
# Measures overlap between predicted box and true box
def compute_iou(pred_box, true_box):

    x1 = max(pred_box[0], true_box[0])
    y1 = max(pred_box[1], true_box[1])
    x2 = min(pred_box[2], true_box[2])
    y2 = min(pred_box[3], true_box[3])

    inter_area = max(0, x2 - x1) * max(0, y2 - y1)

    pred_area = (pred_box[2] - pred_box[0]) * (pred_box[3] - pred_box[1])
    true_area = (true_box[2] - true_box[0]) * (true_box[3] - true_box[1])

    union_area = pred_area + true_area - inter_area

    return inter_area / union_area if union_area > 0 else 0.0


# eval
# Calculates Mean IoU, IoU accuracy, and detection rate
def evaluate_model(model, data_loader, device, score_threshold=0.30, iou_threshold=0.50):

    model.eval()

    ious = []
    found_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, targets in data_loader:

            images = [img.to(device) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                total_images += 1

                true_box = target["boxes"][0].cpu().tolist()
                boxes = output["boxes"].cpu()
                scores = output["scores"].cpu()

                keep = scores >= score_threshold

                if keep.sum() == 0:
                    ious.append(0.0)
                    continue

                best_idx = torch.argmax(scores[keep])
                pred_box = boxes[keep][best_idx].tolist()

                found_predictions += 1
                ious.append(compute_iou(pred_box, true_box))

    mean_iou = sum(ious) / len(ious)
    iou_accuracy = sum(i >= iou_threshold for i in ious) / len(ious)
    detection_rate = found_predictions / total_images

    return mean_iou, iou_accuracy, detection_rate


# MAIN SCRIPT
if __name__ == "__main__":
    torch.manual_seed(34)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    full_dataset = BaseballDetectionDataset(
        frames_dir="/content/extracted_frames",
        xml_dir="/content/annotations",
        augment=False
    )

    print("Total samples:", len(full_dataset))

    if len(full_dataset) == 0:
        raise RuntimeError("No samples found. Check extracted_frames and annotations.")

    # Train/test split with augmentation only on training data
    indices = torch.randperm(
        len(full_dataset),
        generator=torch.Generator().manual_seed(34)
    ).tolist()

    train_size = int(0.80 * len(indices))
    train_indices = indices[:train_size]
    test_indices = indices[train_size:]

    train_dataset_full = BaseballDetectionDataset(
        frames_dir="/content/extracted_frames",
        xml_dir="/content/annotations",
        augment=True
    )

    test_dataset_full = BaseballDetectionDataset(
        frames_dir = "/content/extracted_frames",
        xml_dir = "/content/annotations",
        augment = False
    )

    train_dataset = torch.utils.data.Subset(train_dataset_full, train_indices)
    test_dataset = torch.utils.data.Subset(test_dataset_full, test_indices)

    print("Training samples:", len(train_dataset))
    print("Testing samples:", len(test_dataset))

    train_loader = DataLoader(
        train_dataset,
        batch_size = 2,
        shuffle = True,
        num_workers = 2,
        pin_memory = True,
        collate_fn = collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=2,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        collate_fn=collate_fn
    )

    # Initialize Faster R-CNN
    model = get_model(num_classes=2)
    model.to(device)

    # Optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=0.0001, weight_decay=0.0005)

    epochs = 10
    start_time = time.time()

    best_mean_iou = 0
    best_model_path = "/content/drive/MyDrive/baseball/best_faster_rcnn_baseball.pth"

    # Training loop
    for epoch in range(epochs):
        model.train()

        running_loss = 0.0

        for images, targets in train_loader:
            images = [img.to(device, non_blocking=True) for img in images]

            targets = [
                {k: v.to(device, non_blocking=True) for k, v in t.items()}
                for t in targets
            ]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            running_loss += losses.item()

        avg_train_loss = running_loss / len(train_loader)

        mean_iou, iou_accuracy, detection_rate = evaluate_model(
            model=model,
            data_loader=test_loader,
            device=device,
            score_threshold=0.30,
            iou_threshold=0.50
        )

        if mean_iou > best_mean_iou:
            best_mean_iou = mean_iou
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved new BEST model (Mean IoU: {best_mean_iou:.4f})")

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Mean IoU: {mean_iou:.4f} | "
            f"IoU >= 0.50 Accuracy: {iou_accuracy:.4f} | "
            f"Detection Rate: {detection_rate:.4f}"
        )

    # Final evaluation
    final_mean_iou, final_iou_accuracy, final_detection_rate = evaluate_model(
        model=model,
        data_loader=test_loader,
        device=device,
        score_threshold=0.30,
        iou_threshold=0.50
    )

    print("\nFinal Test Results")
    print("------------------")
    print(f"Final Mean IoU: {final_mean_iou:.4f}")
    print(f"Final IoU >= 0.50 Accuracy: {final_iou_accuracy:.4f}")
    print(f"Final Detection Rate: {final_detection_rate:.4f}")
    print(f"\nBest Mean IoU achieved: {best_mean_iou:.4f}")
    print(f"Best model saved to: {best_model_path}")

    save_path = "/content/drive/MyDrive/baseball/faster_rcnn_baseball.pth"
    torch.save(model.state_dict(), save_path)

    print(f"\nSaved model to {save_path}")
    print(f"Total training time: {time.time() - start_time:.2f} seconds")

Writing faster_rcnn_baseball.py


In [ ]:
# ===== CELL 4: RUN FASTER R-CNN =====

!python faster_rcnn_baseball.py